# VPC 피어링: 다른 리전의 프라이빗 API에 연결

이 실습에서는 Amazon Bedrock AgentCore Gateway를 다른 리전의 **피어링된 VPC에 있는 프라이빗 API Gateway**에 연결합니다. 이는 격리, 규정 준수 또는 근접성을 위해 여러 리전이나 VPC에 서비스를 배포하는 일반적인 엔터프라이즈 패턴입니다.

## 아키텍처

![피어링](./images/peering.png)

## 작동 방식

1. VPC Endpoint가 있는 **프라이빗 API Gateway**가 피어링된 VPC에서 실행됩니다(us-east-1, 10.1.0.0/16).
2. **VPC 피어링 연결**이 서로 다른 리전의 두 VPC를 연결합니다(us-west-2 <-> us-east-1).
3. 두 VPC의 라우팅 테이블 항목이 VPC 간 트래픽을 피어링 연결로 전달합니다.
4. AgentCore Gateway가 소유자 VPC에 **managed Resource Gateway**를 생성합니다(us-west-2, 10.0.0.0/16).
5. Resource Gateway ENI가 API-VPCE DNS를 VPCE의 프라이빗 IP(10.1.x.x)로 확인하고 피어링 연결을 통해 트래픽을 라우팅합니다.

API-VPCE DNS 형식(`{api-id}-{vpce-id}.execute-api.us-east-1.amazonaws.com`)은 **공개적으로 이름 확인이 가능**하며, VPCE의 프라이빗 IP로 확인됩니다.

> **왜 이 방식이 작동할까요?** AWS PrivateLink 기반의 Interface VPC Endpoint는 프라이빗 IP 주소를 가진 ENI를 생성합니다. 라우팅 테이블 기반이라 피어링을 통해 액세스할 수 없는 Gateway VPC Endpoint(S3/DynamoDB)와 달리, 이 IP는 VPC 피어링 연결을 통해 라우팅할 수 있습니다.

VPC egress 및 managed VPC resource에 관한 배경 정보는 [프로젝트 README](../README.md)와 [Managed VPC Resource README 문서](./README.md)를 참조하세요.

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb)을 완료해야 합니다(VPC us-west-2 + AgentCore Gateway 배포 완료).
- **us-east-1 부트스트랩**: 실습 0에서 us-east-1 부트스트랩 셀의 주석을 해제하고 실행합니다.
- **us-east-1 스택 배포**: 실습 0에서 us-east-1 VPC + API Gateway 배포 셀의 주석을 해제하고 실행합니다.

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0의 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

# us-east-1 변수 복원(실습 0에서 배포)
%store -r VPC_USE1_ID
%store -r PEERING_API_ID
%store -r PEERING_API_KEY_ID
%store -r PEERING_VPCE_ID

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION_W = "us-west-2"
REGION_E = "us-east-1"

session_w = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION_W)
session_e = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION_E)

agentcore = session_w.client("bedrock-agentcore-control")
ec2_w = session_w.client("ec2")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session_w.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC (west): {VPC_USW2_ID}")
print(f"VPC (east): {VPC_USE1_ID}")

## 2단계: us-east-1 인프라 확인

다음 스택은 [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb)에서 배포했습니다.

| 스택 | 설명 |
|-------|-------------|
| `VpcegressStack-USEast1` | 퍼블릭, 프라이빗 및 격리 서브넷이 있는 VPC(10.1.0.0/16) |
| `PeeringApigw-USEast1` | 모의 통합 + execute-api VPC Endpoint가 있는 프라이빗 API Gateway |

프라이빗 API Gateway는 [실습 01](./01-getting-started.ipynb)과 동일한 패턴을 사용합니다. API 키로 보호되는 모의 `/health` 및 `/items` 엔드포인트를 제공합니다.

In [ ]:
# API-VPCE DNS: 공개적으로 이름 확인이 가능하며 us-east-1의 VPCE 프라이빗 IP로 확인됨
API_VPCE_DNS = f"{PEERING_API_ID}-{PEERING_VPCE_ID}.execute-api.{REGION_E}.amazonaws.com"

# API 키 값 가져오기
apigw_client = session_e.client("apigateway")
api_key_response = apigw_client.get_api_key(apiKey=PEERING_API_KEY_ID, includeValue=True)
API_KEY_VALUE = api_key_response["value"]

print("=== us-east-1 VPC ===")
print(f"VPC ID:          {VPC_USE1_ID}")
print("\n=== Private API Gateway ===")
print(f"API ID:          {PEERING_API_ID}")
print(f"VPCE ID:         {PEERING_VPCE_ID}")
print(f"API-VPCE DNS:    {API_VPCE_DNS}")

## 3단계: AgentCore Gateway 대상 생성

VPC 피어링 연결, 경로 및 보안 그룹 규칙은 모두 [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb)에서 CDK를 통해 배포했습니다.
- **`VpcPeeringStack`**은 피어링 연결을 생성하고 사용자 지정 리소스를 통해 연결을 수락한 후 두 VPC에 경로를 추가했습니다.
- **`PeeringApigw-USEast1`**은 두 VPC CIDR(10.0.0.0/16 및 10.1.0.0/16)에 대한 인바운드 규칙이 있는 VPCE 보안 그룹을 생성했습니다.

이제 managed VPC resource를 사용하여 Gateway 대상을 생성합니다. Resource Gateway는 **us-west-2 VPC**(소유자 VPC)에 배치되고, 대상 엔드포인트는 **us-east-1**(피어링된 VPC)의 API-VPCE DNS입니다.

트래픽 흐름:
```
AgentCore Gateway -> VPC Lattice -> Resource Gateway ENI (us-west-2)
    -> VPC Peering -> VPCE ENI (us-east-1) -> Private API Gateway
```

In [ ]:
# us-west-2에서 Resource Gateway ENI용 보안 그룹 생성
# 새 보안 그룹은 기본적으로 모든 아웃바운드 트래픽을 허용함
sg = ec2_w.create_security_group(
    GroupName=f"peering-rg-sg-{int(time.time())}",
    Description="Resource Gateway SG for peering lab - allows outbound HTTPS",
    VpcId=VPC_USW2_ID,
)
RG_SG_ID = sg["GroupId"]
print(f"Resource Gateway SG (us-west-2): {RG_SG_ID}")

In [ ]:
# AgentCore에서 API 키 자격 증명 공급자 생성
cred_response = agentcore.create_api_key_credential_provider(
    name="peering-apigw-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL을 피어링된 API-VPCE DNS로 설정
with open("01-managed-vpc-resource/openapi-private-apigw.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{API_VPCE_DNS}/prod"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]
OPENAPI_SCHEMA = json.dumps(openapi_schema)

print(f"Target endpoint: {TARGET_ENDPOINT}")
print(f"Resource Gateway VPC: {VPC_USW2_ID} (us-west-2)")
print(f"API Gateway VPC:      {VPC_USE1_ID} (us-east-1)")

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="peering-apigw",
    description="Private API Gateway in peered VPC (us-east-1) via VPC peering",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [RG_SG_ID],
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 4단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음 AgentCore Gateway를 통해 us-east-1의 프라이빗 API Gateway를 호출합니다. 트래픽은 us-west-2에서 VPC 피어링 연결을 거쳐 us-east-1의 VPCE로 흐릅니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 사용 가능한 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 상태 확인 - us-east-1의 프라이빗 API Gateway에서 GET /health 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "peering-apigw___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check (us-east-1 Private API Gateway via VPC peering):")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "peering-apigw___listItems", "arguments": {}},
        "id": 3,
    },
)
print("Items (from peered VPC):")
print(json.dumps(response.json(), indent=2))

## 정리

1. Gateway 대상 및 자격 증명 공급자를 삭제합니다.
2. Resource Gateway 보안 그룹을 삭제합니다.

> **참고:** VPC 피어링 연결, 경로, 보안 그룹 및 CDK 스택은 [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb)의 정리 과정에서 `cdk destroy VpcPeeringStack PeeringApigw-USEast1 VpcegressStack-USEast1` 명령으로 삭제합니다.

In [ ]:
# # 1단계: Gateway 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 자격 증명 공급자 삭제
# agentcore.delete_api_key_credential_provider(name="peering-apigw-api-key")
# print("Deleted credential provider: peering-apigw-api-key")

In [ ]:
# # 2단계: Resource Gateway 보안 그룹 삭제
# try:
#     ec2_w.delete_security_group(GroupId=RG_SG_ID)
#     print(f"Deleted security group: {RG_SG_ID}")
# except ec2_w.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {RG_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise